# Task 2 — Classificatore manuale: **1R (1-Rule)**

> 📄 Documentazione completa e motivazioni: [`docs/task2.md`](../docs/task2.md)

## 0. Caricamento dei dati

In [1]:
import pandas as pd
import numpy as np

In [2]:
m = pd.read_csv("../data/processed/manuale.csv")
m

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


In [3]:
nominali = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]
numerici = ["age", "campaign"]
print("Nominali:", nominali)
print("Numerici:", numerici)
print("\nDistribuzione classe:")
print(m["y"].value_counts())

Nominali: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
Numerici: ['age', 'campaign']

Distribuzione classe:
y
1    6
0    6
Name: count, dtype: int64


## 1. Come funziona 1R (teoria, Lezione 5)

## 2. Adattamento ai dati — calcolo degli errori per ogni attributo

In [4]:
def regola_1R(serie_attr, y):
    """Restituisce (regole, errori_totali) per un attributo, in modo vettoriale.
    regole: dizionario {valore: classe_assegnata}."""
    regole = y.groupby(serie_attr).agg(lambda s: s.mode()[0]).to_dict()
    tab = pd.crosstab(serie_attr, y)
    errori = int((tab.sum(axis=1) - tab.max(axis=1)).sum())
    return regole, errori

### 2.1 Errori sugli attributi nominali

In [5]:
def _stampa_nom(attr):
    regole, err = regola_1R(m[attr], m["y"])
    print(f"{attr:12s}: {err}/12 errori   regole={regole}")
    return err

errori_attr = pd.Series(nominali, index=nominali).apply(_stampa_nom).to_dict()

job         : 2/12 errori   regole={'admin.': 1, 'blue-collar': 0, 'retired': 1, 'student': 1, 'technician': 1, 'unknown': 0}
marital     : 2/12 errori   regole={'divorced': 1, 'married': 0, 'single': 1}
education   : 3/12 errori   regole={'basic.4y': 1, 'basic.6y': 0, 'basic.9y': 1, 'high.school': 0, 'professional.course': 1, 'university.degree': 0, 'unknown': 0}
housing     : 5/12 errori   regole={'no': 0, 'unknown': 1, 'yes': 0}
loan        : 5/12 errori   regole={'no': 0, 'unknown': 1, 'yes': 0}
contact     : 5/12 errori   regole={'cellular': 1, 'telephone': 0}
poutcome    : 6/12 errori   regole={'failure': 0, 'nonexistent': 0}


### 2.2 Errori sugli attributi numerici (discretizzati per mediana)

In [6]:
def _stampa_num(attr):
    med = m[attr].median()
    binned = (m[attr] > med).map({False: f"<= {med}", True: f"> {med}"})
    regole, err = regola_1R(binned, m["y"])
    print(f"{attr:12s} (mediana={med}): {err}/12 errori   regole={regole}")
    return err

errori_num = pd.Series(numerici, index=numerici).apply(_stampa_num)
errori_attr.update(errori_num.to_dict())

age          (mediana=36.0): 5/12 errori   regole={'<= 36.0': 1, '> 36.0': 0}
campaign     (mediana=2.5): 4/12 errori   regole={'<= 2.5': 1, '> 2.5': 0}


### 2.3 Scelta dell'attributo migliore

In [7]:
errori_ser = pd.Series(errori_attr).sort_values()
print("Errori per attributo:")
errori_ser.to_frame("e").apply(lambda r: print(f"  {r.name:12s}: {int(r['e'])}/12"), axis=1)

best = errori_ser.idxmin()
print(f"\n=> 1R sceglie '{best}' con {errori_ser[best]}/12 errori")
print(f"   Accuratezza sul training: {(12-errori_ser[best])/12:.2%}")

Errori per attributo:
  job         : 2/12
  marital     : 2/12
  education   : 3/12
  campaign    : 4/12
  loan        : 5/12
  contact     : 5/12
  age         : 5/12
  housing     : 5/12
  poutcome    : 6/12

=> 1R sceglie 'job' con 2/12 errori
   Accuratezza sul training: 83.33%


## 3. Implementazione: il classificatore 1R completo

In [8]:
def addestra_1R(df, attributi_nominali, attributi_numerici, y_col="y"):
    def err_attr(a):
        if a in attributi_numerici:
            med = df[a].median()
            binned = (df[a] > med).map({False: "low", True: "high"})
            return regola_1R(binned, df[y_col])[1]
        return regola_1R(df[a], df[y_col])[1]

    attrs = pd.Series(attributi_nominali + attributi_numerici)
    errori = attrs.apply(err_attr); errori.index = attrs
    best = errori.idxmin()

    if best in attributi_numerici:
        med = df[best].median()
        binned = (df[best] > med).map({False: "low", True: "high"})
        regole = df[y_col].groupby(binned).agg(lambda s: s.mode()[0]).to_dict()
        return {"attributo": best, "regole": regole, "soglia": med, "errori": int(errori[best])}
    else:
        regole = df[y_col].groupby(df[best]).agg(lambda s: s.mode()[0]).to_dict()
        return {"attributo": best, "regole": regole, "soglia": None, "errori": int(errori[best])}

In [9]:
def predici_1R(modello, istanza):
    attr = modello["attributo"]
    if modello["soglia"] is not None:          # attributo numerico
        val = "high" if istanza[attr] > modello["soglia"] else "low"
    else:
        val = istanza[attr]
    return modello["regole"].get(val, 0)        # default 0 se valore mai visto

### 3.1 Valutazione in leave-one-out

In [10]:
def _loo_1R(i):
    modello = addestra_1R(m.drop(i).reset_index(drop=True), nominali, numerici)
    return predici_1R(modello, m.iloc[i])

pred_1R = pd.Series(m.index, index=m.index).apply(_loo_1R)
acc_1R = (pred_1R == m["y"]).mean()
print(f"Accuratezza 1R (leave-one-out): {acc_1R:.2%}")

Accuratezza 1R (leave-one-out): 41.67%


---
## Riepilogo

- **1R** sceglie il solo attributo `job` (2/12 errori sul training, 83%).
- In **leave-one-out** l'accuratezza scende sensibilmente: il divario tra training e
  leave-one-out è la prova concreta dell'**overfitting** segnalato dalle slide.
- 1R è il classificatore più semplice e interpretabile: un buon **baseline** da
  confrontare con Naïve Bayes (vedi notebook dedicato).

Le metriche su 12 istanze sono poco affidabili: illustrano il funzionamento, non
giudicano il modello (valutazione seria nei Task 4–5).